# Bonus Challenge

In [1]:
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

# Define the project paths

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / "data"
DELTA_DIR = PROJECT_ROOT / "delta_bonus"

DELTA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root   : {PROJECT_ROOT}")
print(f"Raw directory  : {RAW_DIR}")
print(f"Delta directory: {DELTA_DIR}")

Project root   : C:\lufthansa-de-exercise
Raw directory  : C:\lufthansa-de-exercise\data
Delta directory: C:\lufthansa-de-exercise\delta_bonus


# Create the spark session in combination with delta lake.

In [3]:
builder = (
    SparkSession.builder
    .appName("bonustask-pipeline")
    .master("local[*]")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Define the table configuration

In tables are included its name, csv file, timestamp_column for partitioning and keys for deduplication. 
Some tables have more than 1 key, since they are not unique in that specific table.

TABLE_NAME is a variable, so with its change, another table can be configured.

In [4]:
TABLES = {
    "orders": {
        "csv_file": "olist_orders_dataset.csv",
        "timestamp_col": "order_purchase_timestamp",
        "key": ["order_id"],
    },
    "order_items": {
        "csv_file": "olist_order_items_dataset.csv",
        "timestamp_col": "shipping_limit_date",
        "key": ["order_id", "order_item_id"],
    },
    "order_reviews": {
        "csv_file": "olist_order_reviews_dataset.csv",
        "timestamp_col": "review_creation_date",
        "key": ["review_id", "order_id"],
    },
    "order_payments": {
        "csv_file": "olist_order_payments_dataset.csv",
        "timestamp_col": None,
        "key": ["order_id", "payment_sequential"],
    },
    "customers": {
        "csv_file": "olist_customers_dataset.csv",
        "timestamp_col": None,
        "key": ["customer_id"],
    },
    "products": {
        "csv_file": "olist_products_dataset.csv",
        "timestamp_col": None,
        "key": ["product_id"],
    },
    "sellers": {
        "csv_file": "olist_sellers_dataset.csv",
        "timestamp_col": None,
        "key": ["seller_id"],
    },
}

# value can be changed
TABLE_NAME = "order_items"

if TABLE_NAME not in TABLES:
    raise ValueError(
        f"Unknown table: {TABLE_NAME!r}. "
    )

TABLE_CONFIG = TABLES[TABLE_NAME]
CSV_FILE = TABLE_CONFIG["csv_file"]
TIMESTAMP_COLUMN = TABLE_CONFIG["timestamp_col"]
DEDUP_KEY = TABLE_CONFIG["key"]

INPUT_PATH = RAW_DIR / CSV_FILE

# another folder "processed" is created to write the table. bronze and silver arent used.
OUTPUT_PATH = DELTA_DIR / "processed" / TABLE_NAME

print(f"Selected table: {TABLE_NAME}")
print(f"Input file: {INPUT_PATH}")
print(f"Timestamp column: {TIMESTAMP_COLUMN}")
print(f"Deduplication key: {DEDUP_KEY}")
print(f"Output path: {OUTPUT_PATH}")

Selected table: order_items
Input file: C:\lufthansa-de-exercise\data\olist_order_items_dataset.csv
Timestamp column: shipping_limit_date
Deduplication key: ['order_id', 'order_item_id']
Output path: C:\lufthansa-de-exercise\delta_bonus\processed\order_items


# Reusable functions

In [5]:
PARTITION_COLUMNS = ["year", "month", "day"]

# load a csv file or a delta table
def load_data(spark: SparkSession, path, source_type: str = "csv") -> DataFrame:

    source_type = source_type.lower()

    # multiLine and escape are required for order_reviews as the comment text contains embedded newlines and quotes
    if source_type == "csv":
        return spark.read.csv(
            str(path),
            header=True,
            inferSchema=True,
            multiLine=True,
            escape='"',
        )

    if source_type == "delta":
        return (
            spark.read
            .format("delta")
            .load(str(path))
        )

    raise ValueError(
        "source_type must be either 'csv' or 'delta'"
    )

# parse a timestamp column and add year, month, and day
def add_partition_columns(df: DataFrame, timestamp_column: str) -> DataFrame:

    if timestamp_column not in df.columns:
        raise ValueError(
            f"Column {timestamp_column!r} was not found. "
            f"Available columns: {df.columns}"
        )

    # parse the timestamp column before extracting 
    df = df.withColumn(
        timestamp_column,
        F.to_timestamp(F.col(timestamp_column)),
    )

    return (
        df
        .withColumn("year", F.year(F.col(timestamp_column)))
        .withColumn("month", F.month(F.col(timestamp_column)))
        .withColumn("day", F.dayofmonth(F.col(timestamp_column)))
    )

# appply cleaning and table transformations
def apply_transformations(df: DataFrame, table_name: str, dedup_key=None) -> DataFrame:

    # deduplication uses the key rather than a direct dropDuplicates
    
    if dedup_key:
        df = df.dropDuplicates(dedup_key)

    if table_name == "orders":
        timestamp_columns = [
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]

        # parse the timestamp columns found
        for column_name in timestamp_columns:
            if column_name in df.columns:
                df = df.withColumn(
                    column_name,
                    F.to_timestamp(F.col(column_name)),
                )

        # various column checks
        # order id and customer id for null values, order status for the data format, F.expr to find the delivery days for an order
        df = (
            df
            .filter(F.col("order_id").isNotNull())
            .filter(F.col("customer_id").isNotNull())
            .withColumn( "order_status", F.lower(F.trim(F.col("order_status"))))
            .withColumn( "delivery_time_days", F.expr( "datediff(" "order_delivered_customer_date, " "order_purchase_timestamp" ")"))
            # we use F.when for conditional logic : not_delivered, on_time, late, while checking for null values in order_delivered_customer_date
                
            .withColumn( "delivery_status", F.when( F.col("order_delivered_customer_date").isNull(), "not_delivered")
                .when(F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date"), "on_time")
                .otherwise("late")))

        # Window transformation 
        # creates a window for each customer
        customer_window = (
            Window.partitionBy("customer_id").orderBy("order_purchase_timestamp"))

        # adds a column with an order sequence number
        df = df.withColumn("customer_order_number", F.row_number().over(customer_window))

    # added some transformations for these 2 tables order items and order reviews
    # if the table name is order_items
    # we calculate total item value
    elif table_name == "order_items":
        df = (df.filter(F.col("order_id").isNotNull()).filter(F.col("seller_id").isNotNull()).withColumn("total_item_value",
                F.round(F.expr("price + freight_value"), 2)))

    # if the table name is order_reviews
    elif table_name == "order_reviews":
        df = (df.filter(F.col("review_id").isNotNull()).withColumn("review_comment_length",
                F.when( F.col("review_comment_message").isNotNull(),F.length(F.col("review_comment_message"))).otherwise(0)))

    return df

# save a DataFrame as a Delta table
def save_delta( df: DataFrame, path, partition_columns=None, mode: str = "overwrite") -> None:
# None means that the function does not return a dataFrame, it just writes the data and prints

    # checks for partition columns
    if partition_columns is None:
        partition_columns = []

    missing_columns = []

    for column in partition_columns:
        if column not in df.columns:
            missing_columns.append(column)

    if missing_columns:
        raise ValueError(
            f"Partition columns not found: {missing_columns}"
        )

    # builds the writer
    writer = (df.write.format("delta").mode(mode).option("overwriteSchema", "true"))

    # add the partition
    if partition_columns:
        writer = writer.partitionBy(*partition_columns)

    writer.save(str(path))

    print(f"Delta table saved to: {path}")
    print(f"Partition columns: {partition_columns or 'none'}")

# main function , used to connect all functions above to finalize the delta table
def process_table(spark: SparkSession, table_name: str, input_path, output_path, timestamp_column=None, dedup_key=None, source_type: str = "csv") -> DataFrame:
    
    # load 1 table
    df = load_data(spark=spark, path=input_path, source_type=source_type)
    print(f"Loaded {df.count():,} rows for {table_name} table")

    # Load the partition for that table
    if timestamp_column is not None:
        df = add_partition_columns(
            df=df,
            timestamp_column=timestamp_column,
        )
        partition_columns = PARTITION_COLUMNS
    else:
        partition_columns = []
        
    # load transforamtions
    df = apply_transformations(df=df, table_name=table_name, dedup_key=dedup_key)

    # keep the dataframe in memory
    df = df.cache()

    save_delta(df=df, path=output_path, partition_columns=partition_columns, mode="overwrite")

    return df

# Run the selected table

In [6]:
# running the function with the parameters given
processed_df = process_table(
    spark=spark,
    table_name=TABLE_NAME,
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    timestamp_column=TIMESTAMP_COLUMN,
    dedup_key=DEDUP_KEY,
    source_type="csv",
)

Loaded 112,650 rows for order_items table
Delta table saved to: C:\lufthansa-de-exercise\delta_bonus\processed\order_items
Partition columns: ['year', 'month', 'day']


# Check the result of the table

In [7]:
processed_df.printSchema()
processed_df.show(10, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- total_item_value: double (nullable = true)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+----+-----+---+----------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|year|month|day|total_item_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+--

In [8]:
processed_df.limit(15).toPandas()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,year,month,day,total_item_value
0,0010b2e5201cc5f1ae7e9c6cc8f5bd00,1,5a419dbf24a8c9718fe522b81c69f61a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-09-15 18:04:37,48.90,16.60,2017,9,15,65.50
1,00143d0f86d6fbd9f9b38ab440ac16f5,1,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20 16:07:52,21.33,15.10,2017,10,20,36.43
2,001ab0a7578dd66cd4b0a71f5b6e1e41,1,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63,2018,1,4,42.52
3,00229e4e43f7a7e0b9dd819ad43268d3,1,13fcfc313dfb2217e5ee3000a702f9ef,c3cfdc648177fdbbbb35635a37472c53,2018-04-10 16:50:11,74.90,16.49,2018,4,10,91.39
4,00276d5c3491fbf55305e26891040df9,1,71c89e9d8fa274b017279aa322cd0e19,8f78f0903005064036736c7173a5c2ed,2018-02-27 11:55:57,44.90,23.22,2018,2,27,68.12
5,0028de0ca693a1bb26448916a81105cc,1,059344baebbeaa42fa9f2bbe11b1583e,955fee9216a65b617aa5c0531780ce60,2018-08-21 03:35:17,29.99,15.31,2018,8,21,45.30
6,0030d783f979fbc5981e75613b057344,1,ae27a5524edb2c8dc4656c670f458fb7,8e6cc767478edae941d9bd9eb778d77a,2017-11-29 23:13:38,60.60,17.67,2017,11,29,78.27
7,0032d07457ae9c806c79368d7d9ce96b,1,08279c494018541f71443c07d77560f8,e333046ce6517bd8bb510291d44f0130,2018-03-15 19:28:51,159.00,27.19,2018,3,15,186.19
8,00335b686d693c7d72deeb12f8e89227,1,87b08e712cc4c9fe70984c5a24b29e2f,f00e21b1e91a79653163b7fd8f293ff1,2017-07-28 03:45:26,63.90,16.89,2017,7,28,80.79
9,0035e6b7ade84b3f5b86bd49814793df,1,71a5f1c2a5fd9889ef26b5ac22aec9c6,537eb890efff034a88679788b647c564,2018-02-27 03:31:08,19.90,14.10,2018,2,27,34.00


# Validate the saved delta table

In [9]:
# read data from the transformed delta table from processed folder
saved_df = load_data(
    spark=spark,
    path=OUTPUT_PATH,
    source_type="delta",
)

# check the count of the dataframe present in spark with the transformed table
processed_count  = processed_df.count()
transformed_count = saved_df.count()

print(f"Rows processed in df      : {processed_count :,}")
print(f"Rows saved in delta table : {transformed_count:,}")

Rows processed in df      : 112,650
Rows saved in delta table : 112,650


In [10]:
spark.stop()